# Animal recognition in video

The Claude API takes images, not video. This example decodes a short clip into a handful of
**timestamp-labeled frames**, sends them to Claude as one ordered sequence, and turns the
model's per-frame answer into a **per-species timeline**.

**Requirements:** `ANTHROPIC_API_KEY` (env var or a `.env` at the repo root), a short video clip,
and OpenCV for decoding (installed via `requirements.txt`).

In [ ]:
# Setup
import os
import sys

from dotenv import load_dotenv
from anthropic import Anthropic

# Shared helpers (model, prompt, frame sampling, timeline) live in _video.py. Make it importable
# whether the working directory is this folder or the repo root.
for _p in (".", "video"):
    if os.path.isfile(os.path.join(_p, "_video.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

from _video import (
    MODEL,
    analyze_frames,
    build_timeline,
    fetch_video_to_temp,
    format_timeline,
    sample_frames,
)

load_dotenv()
client = Anthropic()

## 1. Sample frames

Sending every frame would be wasteful — 30 fps over a minute is thousands of images. We keep one
frame every `every_sec` seconds (capped at `max_frames`), downscale each to hold down image-token
cost, and JPEG-encode it. For real footage, smarter sampling (scene-change or motion detection)
would drop near-duplicate and empty frames; fixed-interval keeps this example simple.

In [ ]:
# Point this at a short wildlife clip — a local path or a public URL.
# Need a sample? Royalty-free clips are at pexels.com/videos and pixabay.com/videos.
VIDEO = "your_clip.mp4"

# For a URL, download it first (OpenCV needs a real file path):
# VIDEO = fetch_video_to_temp("https://example.com/clip.mp4")

frames = sample_frames(VIDEO, every_sec=1.0, max_frames=20)
print(f"Sampled {len(frames)} frames at t = {[t for t, _ in frames]}")

# Preview the first sampled frame
from IPython.display import Image as IPyImage

IPyImage(data=frames[0][1])

## 2. Analyze the sequence

All sampled frames go to Claude in a single request, each preceded by a `[frame i | t=Xs]` label
so the model can reason about order and timing. The prompt asks for strict JSON: the species in
each frame plus a short overall summary.

In [ ]:
# One Messages call over the whole frame sequence.
result = analyze_frames(client, frames)
print(result["summary"])

## 3. Build a per-species timeline

From the per-frame JSON we collapse detections into one row per species — when it first and last
appears, which frames, and the largest count seen. Same division of labor as the confidence
rating in the vision example: let the model do perception, do the bookkeeping in plain Python.

In [ ]:
timeline = build_timeline(result)
print(format_timeline(timeline))

# `timeline` is a plain dict you can post-process, e.g.:
# {"Red fox": {"times": [0.0, 2.0], "max_count": 1, "scientific_name": "Vulpes vulpes"}}

## Notes and next steps

- **Re-identification is approximate.** Claude reasons over the frames you send; it does not
  pixel-track individuals. "Is this the same fox in frame 2 and frame 8?" is a soft judgment. For
  robust identity tracking, pair a detector + tracker (e.g. YOLO + ByteTrack) with Claude for
  species ID on the cropped tracks.
- **Cost scales with frames.** Downscaling, sparser sampling, and prompt caching all help. A cheap
  first pass (Haiku, or a local motion filter) can skip empty frames before the detailed pass.
- **Long videos:** process in segments and summarize hierarchically (per-segment, then overall).
- **Audio is invisible** to vision — birdsong and calls need a separate audio model.